In [8]:
# 1. Daily Weather Logger Simulation
import csv
import os
from datetime import datetime

def log_weather_data(city, temp_min, temp_max, condition):
    filename = "weather_data.csv"
    file_exists = os.path.isfile(filename)
    
    # Get current timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Define CSV headers
    fields = ["timestamp", "city", "temperature_min", "temperature_max", "condition"]
    
    # Append or create a new CSV file
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(fields)
            
        writer.writerow([timestamp, city, temp_min, temp_max, condition])
    
    print(f"Successfully logged weather data for {city} at {timestamp}.")

# Example usage:
if __name__ == "__main__":
    log_weather_data("Manila", 26.0, 32.0, "Sunny")
    log_weather_data("Cebu", 25.0, 30.0, "Partly Cloudy")

Successfully logged weather data for Manila at 2026-05-02 22:24:38.
Successfully logged weather data for Cebu at 2026-05-02 22:24:38.


In [9]:
# 2. JSON Weather Exporter
import csv
import json
import os

def export_to_json(city_name):
    csv_filename = "weather_data.csv"
    json_filename = f"{city_name.lower()}_weather.json"
    
    if not os.path.exists(csv_filename):
        print("CSV file not found. Please run the logger simulation first.")
        return
    
    city_data = []
    
    # Read the CSV file and filter by city
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row['city'].lower() == city_name.lower():
                city_data.append(row)
                
    # Write to JSON
    with open(json_filename, mode='w', encoding='utf-8') as json_file:
        json.dump(city_data, json_file, indent=4)
        
    print(f"Successfully exported {len(city_data)} records for {city_name} to {json_filename}.")

# Example usage:
if __name__ == "__main__":
    export_to_json("Manila")

Successfully exported 2 records for Manila to manila_weather.json.


In [10]:
# 3. Forecast Data Visualizer (Animated)
import csv
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def create_forecast_animation(city_name, days=3):
    # Ensure a valid integer input for days <= 7
    days = min(max(1, days), 7)
    csv_filename = "weather_data.csv"
    
    if not os.path.exists(csv_filename):
        print("CSV file not found. Please create 'weather_data.csv' first.")
        return

    dates = []
    min_temps = []
    max_temps = []
    conditions = []

    # Filter data for the requested city
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row['city'].lower() == city_name.lower():
                dates.append(row['timestamp'].split(' ')[0]) # Date only
                min_temps.append(float(row['temperature_min']))
                max_temps.append(float(row['temperature_max']))
                conditions.append(row['condition'])
                
    # Limit to the requested number of forecast days
    dates = dates[:days]
    min_temps = min_temps[:days]
    max_temps = max_temps[:days]
    conditions = conditions[:days]

    if not dates:
        print(f"No data available for city: {city_name}")
        return

    # Initialize the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_xlim(0, len(dates) - 1 if len(dates) > 1 else 1)
    ax.set_ylim(min(min_temps) - 2, max(max_temps) + 2)
    ax.set_title(f"Temperature Forecast for {city_name}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Temperature (°C)")
    
    # Lines
    line_min, = ax.plot([], [], label="Min Temp", color="blue", marker="o")
    line_max, = ax.plot([], [], label="Max Temp", color="red", marker="s")
    
    ax.legend()
    ax.grid(True)

    def init():
        line_min.set_data([], [])
        line_max.set_data([], [])
        return line_min, line_max

    def update(frame):
        # Update function for the animation
        x = range(frame + 1)
        line_min.set_data(x, min_temps[:frame + 1])
        line_max.set_data(x, max_temps[:frame + 1])
        
        # Adding x-ticks sequentially
        ax.set_xticks(range(len(dates)))
        ax.set_xticklabels(dates)
        
        return line_min, line_max

    # Create the animation
    ani = animation.FuncAnimation(fig, update, frames=len(dates), init_func=init, blit=True, repeat=False)

    # Save animation
    output_filename = "forecast_plot.gif"
    ani.save(output_filename, writer='pillow', fps=1)
    
    print(f"Animation saved as {output_filename}")
    plt.close()

# Example usage:
if __name__ == "__main__":
    create_forecast_animation("Manila", days=3)

Animation saved as forecast_plot.gif
